# v5 경계 사례 — 모델이 틀리는 자리와 라벨이 흔들리는 자리

`견적반영` ↔ `계약·질의검토` 혼동은 v4에서 오답의 32%였고, 모델 계열·용량·입력·앙상블·
결정 구조·데이터 40% 증가 어느 것으로도 움직이지 않았다(가이드 02, decisions-06·08).

여기서 묻는 것은 **"왜 안 움직이는가"** 다. 가설은 두 가지다.

1. **모델이 못 배웠다** — 신호는 본문에 있는데 잡아내지 못한다.
2. **정답이 흔들린다** — 본문만으로 결정되지 않는 자리라 라벨 자체가 조건에 따라 바뀐다.

가설 2는 보통 검증하기 어렵지만, 이 저장소에는 **같은 요구사항 1,445건을 프롬프트만
바꿔 라벨링한 `v6`이 있다.** v5와 v6이 갈리는 자리를 "라벨이 흔들리는 자리"로 두고,
모델 오답이 거기에 몰리는지 보면 두 가설을 가를 수 있다.

> 주의 — v6은 프롬프트와 인출 방식이 함께 바뀌었다(decisions-08). 그래서 v6은
> **"정답"이 아니라 "다른 조건에서의 판정"** 이다. 여기서 쓰는 것도 그 용도다.

In [ ]:
"""경계 혼동 127건을 뽑고, 방향과 문서 분포를 본다."""
import collections
import json
import math
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from scripts.evaluation import baselines as B
from scripts.evaluation.folds import make_lodo_folds
from scripts.evaluation.tree_family import LABELS, collect_probabilities
from scripts.labeling.label_dataset import load_label_dataset

BOUNDARY = {"견적반영", "계약·질의검토"}

rows, meta = load_label_dataset()
by_uid = {r["requirement_uid"]: r for r in rows}
# OOF 확률은 어제 트리 판정에 쓴 함수를 그대로 쓴다. 같은 fold, 같은 기준선.
gold, proba, fold_of = collect_probabilities(rows, {"wc": (B.WORD_CHAR_BALANCED, None)})
doc_of_fold = {i: f.test_document for i, f in enumerate(make_lodo_folds(rows))}
document = {uid: doc_of_fold[i] for uid, i in fold_of.items()}

uids = sorted(gold)
pred = {u: LABELS[int(np.argmax(proba["wc"][u]))] for u in uids}
wrong = [u for u in uids if gold[u] != pred[u]]
boundary = [u for u in wrong if {gold[u], pred[u]} == BOUNDARY]

print(f"{meta['dataset_version']} · 평가 {len(uids)}건 · 오답 {len(wrong)}건")
print(f"경계 혼동 {len(boundary)}건 = 오답의 {len(boundary)/len(wrong)*100:.0f}%\n")

print("― 방향: 한쪽으로 쏠리는가 " + "―" * 26)
for (g, p), n in collections.Counter((gold[u], pred[u]) for u in boundary).most_common():
    base = sum(1 for u in uids if gold[u] == g)
    print(f"  정답 {g:<10} → 예측 {p:<10} {n:>4}건  (해당 정답 {base}건의 {n/base*100:.1f}%)")
print("  → 65 대 62. 편향이 아니라 **양방향으로 갈리는 진짜 경계**다.")

print("\n― 문서별 (상위 5개) " + "―" * 30)
hit, tot = collections.Counter(document[u] for u in boundary), collections.Counter(document.values())
share = pd.DataFrame(
    [(d, hit[d], tot[d], hit[d] / tot[d] * 100) for d in tot],
    columns=["문서", "경계 혼동", "평가 건수", "비율(%)"],
).sort_values("비율(%)", ascending=False)
print(share.head(5).round(1).to_string(index=False))
print(f"  전 문서에 퍼져 있다 (최소 {share['비율(%)'].min():.1f}% ~ 최대 {share['비율(%)'].max():.1f}%).")

In [ ]:
"""핵심 — 모델 오답은 라벨이 흔들리는 자리에 몰리는가.

v5와 v6이 갈린 건을 '불안정'으로 둔다. 모델이 맞힌 건 / 경계 밖 오답 / 경계 혼동
세 집단에서 불안정 비율을 재고 Wilson 95% 구간으로 비교한다. 구간이 겹치지 않으면
표본 잡음으로 설명되지 않는다.
"""


def wilson(k, n, z=1.96):
    """작은 표본에서도 0·1 근처가 구간을 벗어나지 않는다(정규근사와 다른 점)."""
    if not n:
        return 0.0, 1.0
    p, d = k / n, 1 + z * z / n
    centre = (p + z * z / (2 * n)) / d
    half = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return centre - half, centre + half


v6 = {}
for line in (ROOT / "data/labels/label_dataset_v6.jsonl").read_text("utf-8").splitlines():
    if line.strip():
        record = json.loads(line)
        v6[record["requirement_uid"]] = record["primary_action"]

unstable = {u for u in uids if u in v6 and v6[u] != gold[u]}
print(f"v5·v6 라벨 불일치 {len(unstable)}/{len(uids)} = {len(unstable)/len(uids)*100:.1f}% (전체)\n")

groups = [
    ("모델이 맞힌 건", [u for u in uids if gold[u] == pred[u]]),
    ("경계 밖 오답", [u for u in wrong if u not in set(boundary)]),
    ("경계 혼동", boundary),
]
table = []
for title, group in groups:
    k = sum(1 for u in group if u in unstable)
    lo, hi = wilson(k, len(group))
    table.append((title, len(group), k, k / len(group) * 100, lo * 100, hi * 100))
frame = pd.DataFrame(table, columns=["집단", "건수", "불안정", "비율(%)", "95%↓", "95%↑"])
print(frame.round(1).to_string(index=False))

low_boundary, high_correct = table[2][4], table[0][5]
print(f"\n  경계 혼동 하한 {low_boundary:.1f}% > 맞힌 건 상한 {high_correct:.1f}%"
      f"  → 구간이 겹치지 않는다")
print("  단조 증가한다: 맞힌 건 < 경계 밖 오답 < 경계 혼동")

agree = sum(1 for u in boundary if u in v6 and v6[u] == pred[u])
in_both = [u for u in boundary if u in unstable]
print(f"\n― 모델의 '오답'이 v6에서는 정답인가 " + "―" * 18)
print(f"  경계 혼동 중 v6가 모델과 같은 쪽: {agree}/{len(boundary)} = {agree/len(boundary)*100:.1f}%")
print(f"  라벨이 흔들린 {len(in_both)}건만 보면: {agree/len(in_both)*100:.1f}%")
print("  → 흔들린 자리에서는 절반 넘게 **모델 쪽이 다른 조건의 정답과 일치**한다.")

In [ ]:
"""무엇이 그 자리를 만드는가 — 신호 겹침, cost_basis, 모델의 확신도."""
print("― 1. blocker와 cost_basis가 둘 다 있는가 " + "―" * 14)
both_boundary = sum(
    1 for u in boundary if by_uid[u]["blockers"] and by_uid[u]["cost_basis"] != "없음"
)
both_all = sum(1 for r in rows if r["blockers"] and r["cost_basis"] != "없음")
print(f"  경계 혼동 {both_boundary}/{len(boundary)} = {both_boundary/len(boundary)*100:.0f}%")
print(f"  전체     {both_all}/{len(rows)} = {both_all/len(rows)*100:.1f}%"
      f"   → {both_boundary/len(boundary)/(both_all/len(rows)):.1f}배 농축")
print("  결정 21은 이럴 때 blocker를 우선하라고 정한다. 모델은 비용 쪽을 따라간다.")

print("\n― 2. cost_basis 값별 경계 혼동률 " + "―" * 20)
counts = collections.Counter(by_uid[u]["cost_basis"] for u in boundary)
basis = pd.DataFrame(
    [(v, counts[v], sum(1 for r in rows if r["cost_basis"] == v)) for v in counts],
    columns=["cost_basis", "경계 혼동", "전체"],
)
basis["비율(%)"] = basis["경계 혼동"] / basis["전체"] * 100
print(basis.sort_values("비율(%)", ascending=False).round(1).to_string(index=False))
print("  `복합`이 28.7%로 가장 높고 `없음`은 1.2%다. **비용 근거가 섞일수록 갈린다.**")

print("\n― 3. 모델은 헷갈려 하는가 " + "―" * 26)
def gap(uid):
    ordered = np.sort(proba["wc"][uid])[::-1]
    return float(ordered[0] - ordered[1])

gaps_boundary = [gap(u) for u in boundary]
gaps_right = [gap(u) for u in uids if gold[u] == pred[u]]
print(f"  1·2위 확률차 중앙값 — 경계 혼동 {np.median(gaps_boundary):.3f} · "
      f"맞힌 건 {np.median(gaps_right):.3f}")
print(f"  경계 혼동 중 확률차 0.1 미만: {sum(g < 0.1 for g in gaps_boundary)/len(gaps_boundary)*100:.0f}%")
print("  → 엉뚱하게 찍는 게 아니라 **두 근거가 팽팽해서** 확률이 갈라지지 않는다.")

In [ ]:
"""가장 팽팽한 사례를 직접 읽는다. 숫자보다 이쪽이 설득력 있다."""
for uid in sorted(boundary, key=gap)[:3]:
    row = by_uid[uid]
    probabilities = {label: f"{value:.2f}" for label, value in zip(LABELS, proba["wc"][uid])}
    mark = "  ← v6는 모델 편" if v6.get(uid) == pred[uid] else ""
    print(f"[{uid}] {row.get('requirement_name', '')[:52]}")
    print(f"  v5 정답 {gold[uid]}  /  모델 {pred[uid]}  /  v6 {v6.get(uid, '—')}{mark}")
    print(f"  확률 {probabilities}   확률차 {gap(uid):.3f}")
    print(f"  blockers={row['blockers']}   cost_basis={row['cost_basis']}")
    print(f"  라벨러 근거: {row.get('reasoning', '')[:190]}")
    print()

print("세 건 모두 근거 문장이 **두 축을 다 적고 한쪽을 골랐다** — 라벨러도 경계에서 저울질했다.")
print("  뒤 두 건은 비용 근거를 대고서 '…어서 blocker는 아니다'라고 명시적으로 유보한다.")
print("  첫 건은 반대로 blocker를 잡고서 '확인 후에도 추가 원가가 소요된다'고 비용도 함께 적는다.")
print("그 저울질이 모델 확률에 0.37/0.37, 0.40/0.41로 그대로 재현된다.")

## 결론 — 가설 2를 지지한다

**모델 오답은 라벨이 흔들리는 자리에 몰린다.**

| 집단 | 라벨 불안정 비율 | Wilson 95% |
|---|---:|---|
| 모델이 맞힌 건 (944) | 19.9% | 17.5~22.6 |
| 경계 밖 오답 (274) | 31.8% | 26.5~37.5 |
| **경계 혼동 (127)** | **47.2%** | **38.8~55.9** |

단조 증가하고, 경계 혼동의 하한(38.8%)이 맞힌 건의 상한(22.6%)보다 높아 **구간이
겹치지 않는다.** 모델이 틀리는 자리와 라벨이 흔들리는 자리는 같은 자리다.

그리고 라벨이 흔들린 경계 혼동 60건 중 **58.3%는 v6가 모델과 같은 쪽으로 갔다.**
그 건들에서 모델의 "오답"은 다른 조건에서 매긴 정답과 일치한다.

### 무엇이 그 자리를 만드는가

- **신호가 둘 다 있다.** 경계 혼동의 43%가 blocker와 cost_basis를 모두 가진다
  (전체 15.3%의 **2.8배**). 결정 21은 blocker를 우선하라고 정하지만, 모델은
  비용 쪽을 따라간다.
- **비용 근거가 섞일수록 갈린다.** `cost_basis=복합`은 28.7%가 경계 혼동인 반면
  `없음`은 1.2%다.
- **모델은 헷갈려 한다.** 1·2위 확률차 중앙값이 0.146으로 맞힌 건(0.321)의 절반 이하고,
  35%는 0.1 미만이다. 엉뚱하게 찍는 것이 아니라 근거가 실제로 팽팽하다.
- **전 문서에 퍼져 있다.** 2.1%~30.5%로 특정 문서의 문제가 아니다.

### ⚠️ 이 결론의 한계 — 순환성 (2026-09-07 추가)

**위 47.2%는 그 자체로 "라벨이 모호하다"의 증거가 되지 못한다.** v6c 프롬프트가 바꾼
것이 하필 **blocker 정의**이고, blocker는 정확히 `견적반영`↔`계약·질의검토` 경계를
가르는 축이다. 경계를 겨냥해 고친 프롬프트로 라벨을 다시 매겼으니, 불일치가 경계에
몰리는 것은 상당 부분 **설계된 결과**다. v5·v6 불일치는 독립적인 모호성 측정이 아니다.

덧붙여 두 라벨은 대등하지도 않다. 각자의 기준에서 평가해도 v6이 낮다
(v6 학습·v6 평가 0.5991 대 v5 0.6395, decisions-08 09-07 03:17). 그래서 "불안정"의
일부는 양쪽이 대등하게 갈린 것이 아니라 **한쪽이 덜 일관된 것**이다.

**그럼에도 나머지 증거는 v6와 무관하게 성립한다** — 신호 겹침 2.8배, 확률차 0.146 대
0.321, 경계 오라클 상한 0.737, 데이터 40% 증가에도 불변인 32%. 결론의 방향은
유지되지만 **근거의 무게에서 47.2%는 빼고 읽어야 한다.**

독립적으로 재려면 **같은 프롬프트로 전수를 한 번 더 라벨링**해야 한다. 그러면 갈리는
건이 "정의가 달라진 것"이 아니라 "같은 정의 아래서도 갈리는 것"이 되어 모호성의 직접
증거가 된다. 현재 동일조건 반복은 40건(37/40 = 92.5%)뿐이라 경계만 떼어 보기에는
부족하다.

### 그래서

이 경계는 **표본이 부족해서** 못 배우는 것이 아니다(데이터 40% 증가로 확인됐다).
"견적에 반영할 것인가, 질의로 물어볼 것인가"는 입찰사의 리스크 감내 수준과 계약
맥락에 달려 있고, 그 정보는 **요구사항 문장에 들어 있지 않다.** 없는 정보를 모델이
복원할 수는 없다.

움직이려면 모델이 아니라 **결정 21 규칙을 그 회색지대에서 더 좁혀야 한다** — 특히
blocker와 cost_basis가 함께 있는 221건(전체의 15.3%)에 대한 하위 규칙이 없다.
그 작업은 사람이 해야 하고, 새 규칙으로 재라벨링하면 이 노트북을 다시 돌려
불안정 비율이 내려갔는지 확인할 수 있다.

관련: `13_label_boundary.ipynb`(v4 경계 분석), `18_decision_structure.ipynb`
(OvR·OvO·캐스케이드로 경계를 못 움직인 기록), `docs/history/decisions-08.md`.